## Loading the Dataset

In [1]:
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:

recipe2M = pd.read_csv('recipes_data.csv')

In [3]:
recipe2M.head()

,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


## Removing unnecessary columns & nulls

In [4]:
recipe2M_cleaned=recipe2M.drop(columns=['link', 'source', 'site'], inplace=False)
recipe2M_cleaned.dropna()

,title,ingredients,directions,NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p..."
...,...,...,...,...
2231137,Sunny's Fake Crepes,"[""1/2 cup chocolate hazelnut spread (recommend...","[""Spread hazelnut spread on 1 side of each tor...","[""chocolate hazelnut spread"", ""marshmallows"", ..."
2231138,Devil Eggs,"[""1 dozen eggs"", ""1 paprika"", ""1 salt and pepp...","[""Boil eggs on medium for 30mins."", ""Then cool...","[""choice"", ""miracle whip"", ""eggs"", ""relish"", ""..."
2231139,Extremely Easy and Quick - Namul Daikon Salad,"[""150 grams Daikon radish"", ""1 tbsp Sesame oil...","[""Julienne the daikon and squeeze out the exce...","[""soy sauce"", ""radish"", ""white sesame seeds"", ..."
2231140,Pan-Roasted Pork Chops With Apple Fritters,"[""1 cup apple cider"", ""6 tablespoons sugar"", ""...","[""In a large bowl, mix the apple cider with 4 ...","[""apple cider"", ""egg"", ""sugar"", ""freshly groun..."


## Removing recipes with directions contatining the word "step"

In [5]:

recipe2M_cleaned = recipe2M_cleaned[~recipe2M_cleaned['directions'].str.contains('step', case=False, na=False)]
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with at most 1 ingredient

In [6]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['ingredients'].apply(lambda x: len([i for i in x if i.strip()]) <= 1)].index, inplace=True)
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with instructions less than 10 characters

In [7]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['directions'].apply(lambda x: not all(len(i.strip()) < 10 for i in x if i.strip()))].index, inplace=True)
recipe2M_cleaned['title'].count()

2206617

## Removing recipes with title less than 4 characters 

In [8]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['title'].apply(lambda x: len(str(x)) < 4 if pd.notnull(x) else False)].index, inplace=True)
recipe2M_cleaned['title'].count()

2206372

## Extract raw ingredients

In [9]:
recipes = recipe2M_cleaned

In [10]:
# Tokenize a string into words
recipes['tokens'] = recipes['NER'].apply(word_tokenize)


In [11]:
#adding customized stop words
irrelevant_words = {
    'fresh', 'frozen', 'thawed', 'raw', 'grated', 'diced', 'chopped', 'minced',
    'powdered', 'sliced', 'ground', 'cooked', 'boiled', 'roasted', 'steamed',
    'baked', 'fried', 'toasted', 'crushed', 'peeled', 'skinned', 'shredded',
    'melted', 'whipped', 'pinch', 'dash', 'handful', 'cup', 'tablespoon',
    'teaspoon', 'liter', 'ml', 'oz', 'lb', 'gram', 'kg', 'quart', 'optional',
    'to taste', 'as needed', 'prepared', 'ready-made', 'store-bought', 'homemade',
    'pre-cooked', 'large', 'small', 'medium', 'whole', 'half', 'quartered',
    'extra', 'light', 'dark', 'white', 'black', 'red', 'green', 'yellow',
    'brown', 'golden', 'sweet', 'bitter', 'spicy', 'mild', 'hot', 'cold',
    'water', 'broth', 'stock', 'sauce', 'seasoning', 'marinade','bite','size'
}
stop_words = set(stopwords.words('english'))
stop_words.update(irrelevant_words)

In [12]:
lemmatizer = WordNetLemmatizer()

# Function to lemmatize nouns
def lemmatize(word, pos):
    if pos.startswith('NN'):  
        return lemmatizer.lemmatize(word, pos='n')
    else:
        return word  


In [13]:
#apply stop words removal and lemmatization
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: [lemmatize(word.lower(), tag) for word, tag in nltk.pos_tag(x) if word.isalnum() and word.lower() not in stop_words]
)

In [14]:
#filter uninque ingredients
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: list(set(x))
)

In [15]:
print(recipes['tokens'])

0          [biscuit, nut, vanilla, rice, sugar, milk, but...
1          [breast, soup, cream, mushroom, beef, chicken,...
2          [powder, salt, cream, pepper, cheese, garlic, ...
3            [soup, cream, gravy, cheese, mushroom, chicken]
4          [peanut, graham, chip, cracker, crumb, sugar, ...
                                 ...                        
2231136    [powder, coconut, salt, carrot, curry, vegetab...
2231137    [butter, tortilla, marshmallow, hazelnut, spre...
2231139               [sesame, salt, seed, oil, soy, radish]
2231140    [unsalted, salt, berry, arbol, neutral, cider,...
2231141    [white, salt, pepper, sausage, cheese, paste, ...
Name: tokens, Length: 2206373, dtype: object


In [16]:
#adding ids to recipes
recipes['recipe_id'] = recipes.index + 1

In [17]:
#reorder the columns
columns = ['recipe_id'] + [col for col in recipes.columns if col != 'recipe_id']
recipes = recipes[columns]

In [18]:
recipes.rename(columns={'tokens': 'raw_ingredients'}, inplace=True)

In [19]:
recipes.head()

,recipe_id,title,ingredients,directions,NER,raw_ingredients
0,1,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""...","[biscuit, nut, vanilla, rice, sugar, milk, but..."
1,2,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream...","[breast, soup, cream, mushroom, beef, chicken,..."
2,3,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[powder, salt, cream, pepper, cheese, garlic, ..."
3,4,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c...","[soup, cream, gravy, cheese, mushroom, chicken]"
4,5,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p...","[peanut, graham, chip, cracker, crumb, sugar, ..."


# Extract Cooking Methods

In [20]:
recipes.head()

,recipe_id,title,ingredients,directions,NER,raw_ingredients
0,1,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""...","[biscuit, nut, vanilla, rice, sugar, milk, but..."
1,2,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream...","[breast, soup, cream, mushroom, beef, chicken,..."
2,3,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[powder, salt, cream, pepper, cheese, garlic, ..."
3,4,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c...","[soup, cream, gravy, cheese, mushroom, chicken]"
4,5,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p...","[peanut, graham, chip, cracker, crumb, sugar, ..."


In [21]:
recipes['directions']

0          ["In a heavy 2-quart saucepan, mix brown sugar...
1          ["Place chipped beef on bottom of baking dish....
2          ["In a slow cooker, combine all ingredients. C...
3          ["Boil and debone chicken.", "Put bite size pi...
4          ["Combine first four ingredients and press in ...
                                 ...                        
2231136    ["Cook the onion in butter in a medium saucepa...
2231137    ["Spread hazelnut spread on 1 side of each tor...
2231139    ["Julienne the daikon and squeeze out the exce...
2231140    ["In a large bowl, mix the apple cider with 4 ...
2231141    ["Preheat the oven to 350.", "In a bowl, mix t...
Name: directions, Length: 2206373, dtype: object

In [22]:
cooking_methods_glossary = [
    "bake", "steam", "fry", "grill", "roast", "boil", "sauté", "poach", "broil", "braise",
    "stew", "smoke", "microwave", "blanch", "deep-fry", "barbecue", "sear", "pressure-cook",
    "simmer", "stir-fry","baste","batter","beat","blend","carmelize","chop","cream","cube",
    "cure","dice","dissolve","drain","fold","granish","grate","grease","julienne","knead",
    "marinate","mash","mince","parboil","pare","peel","pinch","pit","plump","preheat","puree",
    "reduce","saute","scald","sear","shred","sift","skim","slice","thaw","toss","whip"
]

In [23]:
stop_words = set(stopwords.words('english'))

In [24]:
cooking_methods = []

for directions in recipes['directions']:
    if pd.isna(directions):
        cooking_methods.append(None)
    else:
        # Tokenize words
        words = word_tokenize(directions.lower())
        # Remove stopwords and non-alphabetic tokens
        filtered_words = [word for word in words if word not in stop_words and word.isalpha()]

        methods = set(filtered_words).intersection(cooking_methods_glossary)

        cooking_methods.append(list(methods))

recipes['cooking_methods'] = cooking_methods

In [25]:
recipes['cooking_methods'].head(20)

0                                      [boil]
1                               [cream, bake]
2                                          []
3                         [boil, cream, bake]
4                                          []
5     [grease, cream, drain, boil, microwave]
6                               [beat, cream]
7                                      [bake]
8                                    [simmer]
9                         [chop, whip, drain]
10                    [fold, dissolve, drain]
11                                         []
12                 [microwave, fry, barbecue]
13                             [cream, drain]
14                                         []
15                              [cream, boil]
16                                     [bake]
17                                     [toss]
18                                    [cream]
19                               [bake, sift]
Name: cooking_methods, dtype: object

## Clean rating dataset

In [26]:
train_rating= pd.read_csv('core-data-train_rating.csv')
test_rating = pd.read_csv('core-data-test_rating.csv')

In [27]:
rating = pd.concat([train_rating, test_rating], ignore_index=True)

In [28]:
rating.drop(columns=['dateLastModified'], inplace=True)

In [29]:
rating.head()

,user_id,recipe_id,rating
0,5215572,17991,5
1,5215572,170724,4
2,5215572,18045,5
3,3622615,60598,4
4,1313770,47519,5


In [30]:

shuffled_recipe_ids = np.random.permutation(recipes['recipe_id'].values)

# Map the shuffled recipe_ids to the ratings dataset
rating['recipe_id'] = shuffled_recipe_ids[:len(rating)]

In [31]:
rating.head()

,user_id,recipe_id,rating
0,5215572,545201,5
1,5215572,1324400,4
2,5215572,830452,5
3,3622615,972957,4
4,1313770,166591,5


In [32]:
recipes.to_csv('cleanedrecipes.csv', index=False)

In [34]:
rating.to_csv('cleanedrating.csv',index=False)

# Models

In [35]:
cleaned_recipes = pd.read_csv('cleanedrecipes.csv')
cleaned_rating = pd.read_csv('cleanedrating.csv')